In [0]:
storage_account = "stmlopsartifacts001"
storage_key = "HTRoBDjrf6m9Oc8kjlUA9S1qcfLg1KSjHQQb4o1NnPcMrR4gyZkpRqW8I3qtB+g5xxI9ahJ6pzYT+ASt8WM8BA=="

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    storage_key
)

In [0]:
dbutils.fs.ls(f"abfss://ml-data@{storage_account}.dfs.core.windows.net/")

In [0]:
# Databricks notebook source
# =========================================
# 0) PARAMÈTRES
# =========================================

storage_account = "stmlopsartifacts001"
container = "ml-data"

customers_file = "customers.csv"
usage_file = "usage_metrics.csv"
billing_file = "billing_support.csv"

# Mets ici une NOUVELLE clé après rotation
storage_key = "HTRoBDjrf6m9Oc8kjlUA9S1qcfLg1KSjHQQb4o1NnPcMrR4gyZkpRqW8I3qtB+g5xxI9ahJ6pzYT+ASt8WM8BA=="

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    storage_key.strip()
)

base_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net"

customers_path = f"{base_path}/{customers_file}"
usage_path = f"{base_path}/{usage_file}"
billing_path = f"{base_path}/{billing_file}"

bronze_path = f"{base_path}/delta/bronze"
silver_path = f"{base_path}/delta/silver"
gold_path = f"{base_path}/delta/gold"

# COMMAND ----------

# =========================================
# 1) IMPORTS
# =========================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# COMMAND ----------

# =========================================
# 2) TEST D'ACCÈS STORAGE
# =========================================

display(dbutils.fs.ls(base_path))

# COMMAND ----------

# =========================================
# 3) LECTURE DES CSV
# =========================================

df_customers_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(customers_path)
)

df_usage_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(usage_path)
)

df_billing_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(billing_path)
)

print("customers columns:", df_customers_raw.columns)
print("usage columns:", df_usage_raw.columns)
print("billing columns:", df_billing_raw.columns)

display(df_customers_raw.limit(10))
display(df_usage_raw.limit(10))
display(df_billing_raw.limit(10))

# COMMAND ----------

# =========================================
# 4) HELPERS
# =========================================

def col_or_null(df, col_name, data_type):
    if col_name in df.columns:
        return F.col(col_name).cast(data_type).alias(col_name)
    return F.lit(None).cast(data_type).alias(col_name)

def col_or_default(df, col_name, data_type, default_value):
    if col_name in df.columns:
        return F.coalesce(F.col(col_name).cast(data_type), F.lit(default_value)).alias(col_name)
    return F.lit(default_value).cast(data_type).alias(col_name)

# COMMAND ----------

# =========================================
# 5) BRONZE
# =========================================

(
    df_customers_raw
    .withColumn("ingestion_ts", F.current_timestamp())
    .write
    .format("delta")
    .mode("overwrite")
    .save(f"{bronze_path}/customers")
)

(
    df_usage_raw
    .withColumn("ingestion_ts", F.current_timestamp())
    .write
    .format("delta")
    .mode("overwrite")
    .save(f"{bronze_path}/usage_metrics")
)

(
    df_billing_raw
    .withColumn("ingestion_ts", F.current_timestamp())
    .write
    .format("delta")
    .mode("overwrite")
    .save(f"{bronze_path}/billing_support")
)

# COMMAND ----------

# =========================================
# 6) SILVER
# =========================================

bronze_customers = spark.read.format("delta").load(f"{bronze_path}/customers")
bronze_usage = spark.read.format("delta").load(f"{bronze_path}/usage_metrics")
bronze_billing = spark.read.format("delta").load(f"{bronze_path}/billing_support")

# Customers: on s'aligne sur le vrai schéma disponible
df_customers_silver = (
    bronze_customers
    .select(
        col_or_null(bronze_customers, "customer_id", "string"),
        col_or_null(bronze_customers, "industry", "string"),
        col_or_null(bronze_customers, "country", "string"),
        col_or_null(bronze_customers, "nps_score", "double"),
        col_or_null(bronze_customers, "tenure_months", "int"),
        col_or_null(bronze_customers, "monthly_contract_value", "double"),
        col_or_null(bronze_customers, "churn_label", "int"),
        F.col("ingestion_ts")
    )
    .filter(F.col("customer_id").isNotNull())
    .dropDuplicates(["customer_id"])
    .filter(F.col("churn_label").isin(0, 1))
)

df_usage_silver = (
    bronze_usage
    .select(
        col_or_null(bronze_usage, "event_id", "string"),
        col_or_null(bronze_usage, "customer_id", "string"),
        F.to_timestamp(F.col("event_date")).alias("event_ts") if "event_date" in bronze_usage.columns else F.lit(None).cast("timestamp").alias("event_ts"),
        col_or_default(bronze_usage, "login_count_30d", "int", 0),
        col_or_default(bronze_usage, "feature_a_usage_30d", "int", 0),
        col_or_default(bronze_usage, "feature_b_usage_30d", "int", 0),
        col_or_default(bronze_usage, "storage_used_gb", "double", 0.0),
        col_or_default(bronze_usage, "api_calls_30d", "int", 0),
        F.col("ingestion_ts")
    )
    .filter(F.col("customer_id").isNotNull())
    .dropDuplicates(["event_id"])
)

df_billing_silver = (
    bronze_billing
    .select(
        col_or_null(bronze_billing, "record_id", "string"),
        col_or_null(bronze_billing, "customer_id", "string"),
        F.to_timestamp(F.col("record_date")).alias("record_ts") if "record_date" in bronze_billing.columns else F.lit(None).cast("timestamp").alias("record_ts"),
        col_or_default(bronze_billing, "invoice_amount", "double", 0.0),
        col_or_default(bronze_billing, "paid_on_time", "int", 1),
        col_or_default(bronze_billing, "days_late", "int", 0),
        col_or_default(bronze_billing, "support_tickets_90d", "int", 0),
        col_or_default(bronze_billing, "csat_score", "double", 5.0),
        F.col("ingestion_ts")
    )
    .filter(F.col("customer_id").isNotNull())
    .dropDuplicates(["record_id"])
)

(
    df_customers_silver.write
    .format("delta")
    .mode("overwrite")
    .save(f"{silver_path}/customers")
)

(
    df_usage_silver.write
    .format("delta")
    .mode("overwrite")
    .save(f"{silver_path}/usage_metrics")
)

(
    df_billing_silver.write
    .format("delta")
    .mode("overwrite")
    .save(f"{silver_path}/billing_support")
)

# COMMAND ----------

# =========================================
# 7) GOLD
# =========================================

silver_customers = spark.read.format("delta").load(f"{silver_path}/customers")
silver_usage = spark.read.format("delta").load(f"{silver_path}/usage_metrics")
silver_billing = spark.read.format("delta").load(f"{silver_path}/billing_support")

usage_gold = (
    silver_usage
    .withColumn(
        "rn",
        F.row_number().over(
            Window.partitionBy("customer_id").orderBy(F.col("event_ts").desc_nulls_last())
        )
    )
    .filter(F.col("rn") == 1)
    .drop("rn", "event_id", "ingestion_ts")
)

billing_gold = (
    silver_billing
    .withColumn(
        "rn",
        F.row_number().over(
            Window.partitionBy("customer_id").orderBy(F.col("record_ts").desc_nulls_last())
        )
    )
    .filter(F.col("rn") == 1)
    .drop("rn", "record_id", "ingestion_ts")
)

df_gold = (
    silver_customers.alias("c")
    .join(usage_gold.alias("u"), on="customer_id", how="left")
    .join(billing_gold.alias("b"), on="customer_id", how="left")
    .select(
        F.col("customer_id"),
        F.col("industry"),
        F.col("country"),
        F.col("nps_score"),
        F.col("tenure_months"),
        F.col("monthly_contract_value"),
        F.coalesce(F.col("u.login_count_30d"), F.lit(0)).alias("login_count_30d"),
        F.coalesce(F.col("u.feature_a_usage_30d"), F.lit(0)).alias("feature_a_usage_30d"),
        F.coalesce(F.col("u.feature_b_usage_30d"), F.lit(0)).alias("feature_b_usage_30d"),
        F.coalesce(F.col("u.storage_used_gb"), F.lit(0.0)).alias("storage_used_gb"),
        F.coalesce(F.col("u.api_calls_30d"), F.lit(0)).alias("api_calls_30d"),
        F.coalesce(F.col("b.invoice_amount"), F.lit(0.0)).alias("invoice_amount"),
        F.coalesce(F.col("b.paid_on_time"), F.lit(1)).alias("paid_on_time"),
        F.coalesce(F.col("b.days_late"), F.lit(0)).alias("days_late"),
        F.coalesce(F.col("b.support_tickets_90d"), F.lit(0)).alias("support_tickets_90d"),
        F.coalesce(F.col("b.csat_score"), F.lit(5.0)).alias("csat_score"),
        F.col("churn_label")
    )
    .withColumn(
        "is_high_value_customer",
        F.when(F.coalesce(F.col("monthly_contract_value"), F.lit(0.0)) >= 120, 1).otherwise(0)
    )
    .withColumn(
        "late_payment_risk",
        F.when(F.coalesce(F.col("days_late"), F.lit(0)) > 10, 1).otherwise(0)
    )
    .withColumn(
        "low_engagement_risk",
        F.when(F.coalesce(F.col("login_count_30d"), F.lit(0)) < 5, 1).otherwise(0)
    )
    .withColumn(
        "support_risk",
        F.when(F.coalesce(F.col("support_tickets_90d"), F.lit(0)) >= 3, 1).otherwise(0)
    )
)

display(df_gold.limit(20))

(
    df_gold.write
    .format("delta")
    .mode("overwrite")
    .save(f"{gold_path}/churn_training_dataset")
)

# COMMAND ----------

# =========================================
# 8) VALIDATION
# =========================================

gold_df = spark.read.format("delta").load(f"{gold_path}/churn_training_dataset")

print("Nombre de lignes gold:", gold_df.count())
gold_df.groupBy("churn_label").count().orderBy("churn_label").show()

display(gold_df)